# 1. To sending all results at Google_Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 2. Setup

In [ ]:
!pip install -qU transformers==4.48.3 datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0 wandb
!pip install -qU json-repair==0.29.1 faker==35.2.0
!pip install -qU vllm==0.7.2 locust

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done


In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
!cd LlamaFactory && pip install -e .
!pip install -qU torchao

Cloning into 'LlamaFactory'...
remote: Enumerating objects: 729, done.
remote: Counting objects: 100% (729/729), done.
remote: Compressing objects: 100% (570/570), done.
remote: Total 729 (delta 160), reused 435 (delta 90), pack-reused 0 (from 0)
Receiving objects: 100% (729/729), 5.49 MiB | 11.89 MiB/s, done.
Resolving deltas: 100% (160/160), done.
Obtaining file:///content/LlamaFactory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import transformers
print("Transformers:", transformers.__version__)

In [ ]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('wandb_key'))
hf_token = userdata.get('hug')
!huggingface-cli login --token {hf_token}

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: melalfy877 (melalfy877-microsoft-office) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Hint: A new version of huggingface_hub (1.31.0) is available! You are using version 1.29.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



# 3. Imports

In [ ]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 2.0 MB/s eta 0:00:00
Using device: cpu


In [ ]:
data_dir = "/content/drive/MyDrive/llm_finetune_arabic_news"

base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
def parse_json(text: str):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

# 3. Tasks

In [ ]:
story = """
ذكرت مجلة فوربس أن العائلة تلعب دورا محوريا في تشكيل علاقة الأفراد بالمال،
 حيث تتأثر هذه العلاقة بأنماط السلوك المالي المتوارثة عبر الأجيال.

التقرير الذي يستند إلى أبحاث الأستاذ الجامعي شاين إنيت حول
الرفاه المالي يوضح أن لكل شخص "شخصية مالية" تتحدد وفقا لطريقة
 تفاعله مع المال، والتي تتأثر بشكل مباشر بتربية الأسرة وتجارب الطفولة.

 الأبعاد الثلاثة للعلاقة بالمال
بحسب الدراسة، هناك ثلاثة أبعاد رئيسية تشكّل علاقتنا بالمال:

الاكتساب (A): يميل الأفراد الذين ينتمون لهذا
 البعد إلى اعتبار المال سلعة قابلة للجمع، حيث يرون
في تحقيق الثروة هدفا بحد ذاته. والجانب السلبي لهذا
 النمط هو إمكانية التحول إلى هوس بالثروة أو العكس،
 أي رفض تام لاكتساب المال باعتباره مصدرا للفساد.

الاستخدام (U): يرى هؤلاء الأشخاص المال أداة للتمتع بالحياة، حيث يربطون قيمته بقدرته على توفير
المتعة والراحة. ومع ذلك، قد يصبح
البعض مدمنا على الإنفاق، في حين يتجه آخرون إلى التقشف المفرط خوفا من المستقبل.

الإدارة (M): أصحاب هذا النمط يعتبرون المال مسؤولية تتطلب التخطيط الدقيق. لكن في بعض الحالات،
 قد يتحول الأمر إلى هوس مفرط بإدارة الإنفاق، مما يؤثر سلبا على العلاقات الشخصية.

 كيف تؤثر العائلة على علاقتنا بالمال؟
يشير التقرير إلى أن التجارب الأسرية تلعب دورا رئيسيا في تحديد
 "الشخصية المالية" لكل فرد، على سبيل المثال، إذا كان أحد الوالدين يعتمد على المال
كمكافأة للسلوك الجيد، فقد يتبنى الطفل لاحقا النمط نفسه في حياته البالغة.

لتحليل هذه التأثيرات بشكل دقيق، طورت رابطة العلاج المالي
(Financial Therapy Association) أداة تسمى مخطط الجينوم المالي (Money Genogram)،
وهو نموذج يُستخدم لتحديد الأنماط المالية داخل العائلة.

تتضمن هذه الأداة:

رسم شجرة عائلية.
تصنيف أفراد العائلة وفقا للأبعاد الثلاثة للعلاقة بالمال (A ،U ،M).
تحديد ما إذا كان السلوك المالي لكل فرد صحيا (+) أو غير صحي (-).
على سبيل المثال، إذا نشأ شخص في عائلة
اعتادت على الإنفاق المفرط، فقد يكون لديه ميل قوي إلى اتباع النمط نفسه،
 أو العكس تماما، حيث يصبح مقتصدا بشكل مبالغ فيه كرد فعل نفسي.
"""

## 3.1 Details Extraction

In [ ]:
StoryCategory = Literal[
    "politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"]

EntityType = Literal[
    "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"]



class Entity(BaseModel):
    entity_value: str = Field(..., description="The actual name or value of the entity.")
    entity_type: EntityType = Field(..., description="The type of recognized entity.")



class NewsDetails(BaseModel):
    story_title: str = Field(..., min_length=10, max_length=200,
                             description="A fully informative and SEO optimized title of the story.")

    story_keywords: List[str] = Field(..., min_items=2,
                                      description="Relevant keywords associated with the story.")

    story_summary: List[str] = Field(
                                    ..., min_items=2, max_items=5,
                                    description="Summarized key points about the story (2-5 points)."
                                )

    story_category: StoryCategory = Field(...,min_items = 1 ,max_items = 5, description="Category of the news story.")

    story_entities: List[Entity] = Field(..., min_items=1, max_items=10,
                                        description="List of identified entities in the story.")

/tmp/ipykernel_700/1064678029.py:41: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_keywords: List[str] = Field(..., min_items=1,
/tmp/ipykernel_700/1064678029.py:44: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field(
/tmp/ipykernel_700/1064678029.py:44: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field(
/tmp/ipykernel_700/1064678029.py:51: PydanticDeprecatedSince20: `min_item

In [ ]:
details_extraction_messages = [
   {
       "role": "system",
       "content": "\n".join([
            "You are an NLP data paraser.",
            "You will be provided by an Arabic text associated with a Pydantic scheme.",
            "Generate the ouptut in the same story language.",
            "You have to extract JSON details from text according the Pydantic details.",
            "Extract details as mentioned in text.",
            "Do not generate any introduction or conclusion."

       ])
   },


   {
       "role": "user",
       "content": "\n".join([
          "## story:",
          story.strip(),
           "",

          "## Pydantic Details:",
          json.dumps(
                NewsDetails.model_json_schema(), ensure_ascii = False
          ),
          "",


          "## story Details",
          "```json"


       ])
   }

]

## 3.2 Translaion

In [ ]:
class TranslatedStory(BaseModel):
    translated_title: str = Field(..., min_length=10, max_length=300,
                                  description="Suggested translated title of the news story.")
    translated_content: str = Field(..., min_length=10,
                                    description="Translated content of the news story.")

targeted_lang = "English"



translation_messages = [
    {
        "role": "system",
        "content": "\n".join([
            "You are a professional translator.",
            "You will be provided by an Arabic text.",
            "You have to translate the text into the `Targeted Language`.",
            "Follow the provided Scheme to generate a JSON",
            "Do not generate any introduction or conclusion."
        ])
    },


    {
        "role": "user",
        "content":  "\n".join([
            "## Story:",
            story.strip(),
            "",

            "## Pydantic Details:",
            json.dumps( TranslatedStory.model_json_schema(), ensure_ascii=False ),
            "",

            "## Targeted Language:",
            targeted_lang,
            "",

            "## Translated Story:",
            "```json"

        ])
    }
]


In [ ]:
NewsDetails.model_json_schema()

# 4. Evaluation

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype= torch_dtype )

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

# 4.1 Evaluation Summarization

In [ ]:
text = tokenizer.apply_chat_template(
    details_extraction_messages,
    tokenize=False,
    add_generation_prompt=True
)

In [ ]:
text

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
print(response)

## 4.2 Evaluation Translation

In [ ]:
text = tokenizer.apply_chat_template(
    translation_messages,
    tokenize=False,
    add_generation_prompt=True
)

In [ ]:
text

In [ ]:
model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
print(response)

# 5. Evaluate by DeepSeek

In [ ]:
from openai import OpenAI
from google.colab import userdata
import httpx

deepseek_model_id = 'deepseek-ai/DeepSeek-R1:novita'

deepseek = OpenAI(
    base_url = "https://router.huggingface.co/v1",
    api_key = userdata.get('deepseek_key'),

)

chat_completion = deepseek.chat.completions.create(
    model=deepseek_model_id,
    messages=details_extraction_messages,
    temperature=0.2,
    max_tokens=1024,
    stream=False
)

print(chat_completion.choices[0].message.content)

In [ ]:
chat_completion = open_client.chat.completions.create(
    messages=translation_messages,
    model=deepseek_model_id,
    temperature=0.2,
)

print(chat_completion.choices[0].message.content)

```json
{
  "translated_title": "Family's Role in Shaping Financial Personality",
  "translated_content": "Forbes magazine noted that family plays a pivotal role in shaping individuals' relationship with money, as this relationship is influenced by inherited financial behavior patterns across generations.\n\nThe report, based on research by university professor Shane Innet on financial well‑being, explains that each person has a \"financial personality\" determined by how they interact with money, which is directly affected by family upbringing and childhood experiences.\n\nThree Dimensions of the Relationship with Money\nAccording to the study, there are three main dimensions that shape our relationship with money:\n\nAcquisition (A): Individuals who belong to this dimension tend to view money as a collectible commodity, seeing wealth accumulation as an end in itself. The downside of this pattern is the potential to become obsessed with wealth or, conversely, to reject acquiring money

# 6. Knowladge Distillation

In [ ]:
raw_data_path = "/gdrive/MyDrive/llm_finetune_arabic_news/datasets/news-sample.jsonl"

raw_data = []
for line in open(raw_data_path):
  if line.strip() == "":
     continue

  raw_data.append(
        json.loads(line.strip()))


random.Random(101).shuffle(raw_data)

print(f"Raw data: {len(raw_data)}")


Raw data: 2400


In [ ]:
raw_data[0]

{'id': 975,
 'title': 'ما تقدمه فلسطين للعالم.. معرض لآمال وآلام شعبها في باريس',
 'description': 'يواصل المعهد العربي في باريس استقبال زواره في معرض “ما تقدمه فلسطين للعالم” لإطلاعهم على الإرث الثقافي والفني للفلسطينيين، من خلال أعمال فنية لآمالهم، وصور لواقعهم الأليم تحت الاحتلال.',
 'content': 'يواصل المعهد العربي في باريس استقبال زواره في معرض ما تقدمه فلسطين للعالم لإطلاعهم على الإرث الثقافي والفني للفلسطينيين؛ من خلال أعمال فنية لآمالهم وصور لواقعهم الأليم تحت الاحتلال. \n ويرى رئيس المعهد جاك لانغ -الذي أُعيد انتخابه قبل أيام للدورة الرابعة- ما يحدث في غزة حاليا جراء العدوان الإسرائيلي أنه كارثة. \n والمعهد هو مركز ثقافي وواجهة دبلوماسية يديرها لانغ منذ 2013 ويقع على ضفة نهر السين في باريس. \n وأشار لانغ، الذي شغل سابقا منصب وزير الثقافة بفرنسا، إلى أن المعرض هو إهداء للشعب الفلسطيني، ومُدّد ليستقبل مزيدا من الزوار حتى 31 ديسمبركانون الأول الجاري. \n ويضم المعرض، الذي افتُتح أواخر مايوأيار الماضي، حسب لانغ العديد من المعارض الفرعية عن فلسطين وعن غزة بالتحديد، من بينها معرض الصور

# 7. Format Finetuning Datasets

## 7.1 Configs

In [ ]:
from openai import OpenAI
deepseek_model = 'deepseek-ai/DeepSeek-R1:novita'

deepseek = OpenAI(
    base_url = "https://router.huggingface.co/v1",
    api_key = userdata.get('deepseek_key'),

)

In [ ]:
def parse_json(text):
  try:
    return json_repair.loads(text)
  except:
    return None

In [ ]:
prompt_tokens = 0
completion_tokens = 0

save_to = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets/sft2.jsonl"

ix= 0

for story in tqdm(raw_data):

  sample_details_extraction_messages = [
    {
        "role":"system",
        "content":"\n".join([
            "you are an NLP data paraser.",
            "you will be provided by an Arabic text associated with a pydantic schema.",
            "generate the output in the same story language.",
            "you have to extract JSON details from text according the pydantic details.",
            "Extract details as mentiones in text",
            "do not generate any intoduction or conclusion."
        ])
    },
    {

    "role":"user",
    "content":"\n".join([
        "## story",
        story['content'].strip(),"",

        "pydantic details:",
        json.dumps(NewsDetails.model_json_schema() , ensure_ascii=False),
        "",
        "## story Details:",
        "```json"
    ])

      }]


  response = deepseek.chat.completions.create(
      messages = sample_details_extraction_messages,
      model = deepseek_model,
      temperature = 0.2,
      )

  if response.choices[0].finish_reason != "stop":
    continue

  llm_response = response.choices[0].message.content
  llm_response = parse_json(llm_response)

  if llm_response is None:
    continue

  with open(save_to , "a" , encoding = "utf-8") as dest:
    dest.write(json.dumps({
        "id":ix,
        "story":story['content'].strip(),
        "task":"Extrat the story details into a JSON.",
        "output_schema":json.dumps(NewsDetails.model_json_schema() , ensure_ascii=False),
        "response":llm_response,

    },ensure_ascii=False , default =str) + "\n")

    ix += 1
    prompt_tokens += response.usage.prompt_tokens
    completion_tokens += response.usage.completion_tokens


    print(f"iteration number {ix} number of prompt tokens {prompt_tokens} number of completion tokens {completion_tokens}")


    if(ix % 5) == 0:
      break

In [ ]:
with open(save_to , 'r' ,encoding='utf-8') as src:
  for line in src:
    print(line)
    break

In [ ]:
prompt_tokens = 0
completion_tokens = 0

save_to_2 = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets/sft3.jsonl"

ix= 0
targeted_lang = "English"
for story in tqdm(raw_data):

  sample_translation_message = [

              {
                  "role":"system",
                  "content" : "\n".join([
              "You are a professional translator.",
              "You will be provided by an Arabic text.",
              "You have to translate the text into the `Targeted Language`.",
              "Follow the provided Scheme to generate a JSON",
              "Do not generate any introduction or conclusion."]
                  )
              },
              {
                  "role":"user",
                  "content" : "\n".join([
                      "## story:",story["content"].strip(),"",


                      "## pydantic details:",
                      json.dumps(TranslateStory.model_json_schema(), ensure_ascii = False),
                      "",

                      "## target language:",targeted_lang,

                      "## translated story:",
                      "```json"
                  ])
              }]


  response = deepseek.chat.completions.create(
      messages = sample_translation_message,
      model = deepseek_model,
      temperature = 0.2,
      )

  if response.choices[0].finish_reason != "stop":
    continue

  llm_response = response.choices[0].message.content
  llm_response = parse_json(llm_response)

  if llm_response is None:
    continue

  with open(save_to_2 , "a" , encoding = "utf-8") as dest:
    dest.write(json.dumps({
        "id":ix,
        "story":story['content'].strip(),
        "task":f"You have to translate the story content into {targeted_lang} associated with a title into a JSON.",
        "output_schema":json.dumps(TranslateStory.model_json_schema() , ensure_ascii=False),
        "response":llm_response,

    },ensure_ascii=False , default =str) + "\n")

    ix += 1
    prompt_tokens += response.usage.prompt_tokens
    completion_tokens += response.usage.completion_tokens


    print(f"iteration number {ix} number of prompt tokens {prompt_tokens} number of completion tokens {completion_tokens}")


    if(ix % 2) == 0:
      break

In [ ]:
with open(save_to_2 , 'r' ,encoding='utf-8') as src:
  for line in src:
    print(line)
    break

## 7.2 Summary

In [ ]:
sft_data_path = "/gdrive/MyDrive/projects/llm_finetune_arabic_news/datasets/sft.jsonl"

In [ ]:
llm_finetunning_data = []

system_message = "\n".join([
    "You are an NLP data parser.",
    "You will be provided by an Arabic text associated with a Pydantic scheme.",
    "Generate the output in the same story language.",
    "You have to extract JSON details from text according to the Pydantic details.",
    "Extract details as mentioned in text.",
    "Do not generate any introduction or conclusion."
])

for line in open(sft_data_path):
    if line.strip() == "":
        continue

    rec = json.loads(line.strip())

    llm_finetunning_data.append({
        "system": system_message,
        "instruction": "\n".join([
            "# Story:",
            rec["story"],

            "# Task:",
            rec["task"],

            "# Output Scheme:",
            rec["output_scheme"],
            "",

            "# Output JSON:",
            "```json"

        ]),
        "input": "",
        "output": "\n".join([
            "```json",
            json.dumps(rec["response"], ensure_ascii=False, default=str),
            "```"
        ]),
        "history": []
    })

random.Random(101).shuffle(llm_finetunning_data)

In [ ]:
llm_finetunning_data

[{'system': 'You are an NLP data parser.\nYou will be provided by an Arabic text associated with a Pydantic scheme.\nGenerate the output in the same story language.\nYou have to extract JSON details from text according to the Pydantic details.\nExtract details as mentioned in text.\nDo not generate any introduction or conclusion.',
  'instruction': '# Story:\nربما كان متوقعًا أن يتراجع حزب العدالة والتنمية الحاكم في تركيا في نتائج الانتخابات المحلية الأخيرة لعدة أسباب، لكن النتائج الأولية أظهرت تراجعًا كبيرًا غير متوقع وخسارة مدوية له، حيث تقدم عليه حزب الشعب الجمهوري لأول مرة، ولم يكتفِ بالاحتفاظ ببلديات بعض المدن الكبرى، وإنما ضم لها مدنًا ومحافظات إضافية في مشهد فوز كاسح شكّل صدمة للكثيرين. \n وَفق النتائج الأولية غير الرسمية، تقدم حزب الشعب الجمهوري باقي الأحزاب وتفوق لأول مرة على العدالة والتنمية منذ تأسيس الأخير، محققًا نسبة أصوات بلغت 37.5 بينما حل حزب العدالة والتنمية ولأول مرة في المركز الثاني بنسبة تصويت 35.6، والرفاه مجددًا في المركز الثالث بنسبة 6.1. \n وقد فاز الشعب الجمهو

In [ ]:
data_dir = r"/gdrive/MyDrive/llm_finetune_arabic_news/datasets"

train_dample_sz = 2600

train_ds = llm_finetunning_data[:train_dample_sz]
eval_ds = llm_finetunning_data[train_dample_sz:]

os.makedirs(join(data_dir,"llamafactory-finetune-data") , exist_ok = True)

with open(join(data_dir,"llamafactory-finetune-data","train.json"),'w') as dest:
  json.dump(train_ds , dest , ensure_ascii=False , default=str)

with open(join(data_dir,"llamafactory-finetune-data","val.json"),'w') as dest:
  json.dump(eval_ds , dest , ensure_ascii=False , default=str)

## 7.3 Translation

In [ ]:
xsft_data_path = "/gdrive/MyDrive/llm_finetune_arabic_news/datasets/xsft.jsonl"

In [ ]:
llm_finetunning_data_t = []

system_message = "\n".join([
     "You are a professional Translator.",
    "Follow the provided `Task` by the user and the `Output Scheme` to generate the `Output JSON`.",
    "Do not generate any introduction or conclusion."
])

for line in open(sft_data_path):
  if line.strip() == '':
    continue

  rec = json.loads(line.strip())

  llm_finetunning_data_t.append({
      "system":system_message,
      "instruction" :"\n".join([
      "# Story:",rec['story'],

      "# Task:" , rec['task'],

      "# Output Scheme:",rec['output_scheme'],"",

      "# Output JSON:",
      "```json",
      ]),
      "input":"",
      "output":"\n".join([
          "```json",
     json.dumps(rec['response'], ensure_ascii=False),
      "```"
      ]) ,
        "history": []
  })

In [ ]:
len(llm_finetunning_data_t)

In [ ]:
data_dir = r"/gdrive/MyDrive/llm_finetune_arabic_news/datasets"

train_dample_sz = 2600

train_ds = llm_finetunning_data_t[:train_dample_sz]
eval_ds = llm_finetunning_data_t[train_dample_sz:]

os.makedirs(join(data_dir,"translate_llamafactory-finetune-data") , exist_ok = True)

with open(join(data_dir,"translate_llamafactory-finetune-data","train.json"),'w') as dest:
  json.dump(train_ds , dest , ensure_ascii=False , default=str)

with open(join(data_dir,"translate_llamafactory-finetune-data","val.json"),'w') as dest:
  json.dump(eval_ds , dest , ensure_ascii=False , default=str)

# 8. Fine Tuning

In [ ]:
%%writefile /content/LlamaFactory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: "Qwen/Qwen2.5-1.5B-Instruct"
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 64
lora_target: all

### dataset
dataset: news_finetune_train
eval_dataset: news_finetune_val
template: qwen
cutoff_len: 3500
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16
dataloader_num_workers: 4

### output
output_dir: "/gdrive/MyDrive/llm_finetune_arabic_news/models"
logging_steps: 10
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000


### eval
# val_size = 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 500


report_to: wandb
run_name: newsx-finetune-llamafactory

#push_to_hub: true
#hub_model_id: "alfy04/news-analyzer"
#hub_private_repo: true
#hub_strategy: checkpoint

Overwriting /content/LlamaFactory/examples/train_lora/news_finetune.yaml


In [ ]:
!cd LlamaFactory/ && llamafactory-cli train /content/LlamaFactory/examples/train_lora/news_finetune.yaml

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[WARNING|2026-09-13 13:39:07] llamafactory.hparams.parser:149 >> We recommend enable mixed precision training.
[INFO|2026-09-13 13:39:07] llamafactory.hparams.parser

# 9. New Evaluation

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = None
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:

finetuned_model_id = "/gdrive/MyDrive/llm_finetune_arabic_news/models"
model.load_adapter(finetuned_model_id)

ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/news-analyzer'

In [ ]:
from peft import PeftModel

finetuned_model_id = "/gdrive/MyDrive/llm_finetune_arabic_news/models"
model = PeftModel.from_pretrained(
    model,
    finetuned_model_id,
    device_map="auto"
)

## 9.1 Translation

In [ ]:
def generate_resp(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

response = generate_resp(translation_messages)

In [ ]:
parse_json(response)

{'translated_title': "Forbes Magazine Reveals Family Plays a Central Role in Forming Individuals' Financial Relationships",
 'translated_content': "According to Forbes magazine, family plays a crucial role in shaping individuals' financial relationships, as these relationships are influenced by inherited behavioral patterns across generations."}

## 9.2 Details Extraction

In [ ]:
def generate_resp(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

response = generate_resp(details_extraction_messages)



In [ ]:
parse_json(response)

# 10. Cost Estimation

In [ ]:
from tqdm.auto import tqdm
from faker import Faker
import random
from datetime import datetime

start_time = datetime.now()
fake = Faker('ar')

input_tokens = 0
output_tokens = 0

for i in tqdm(range(30)):
    prompt = fake.text(max_nb_chars=random.randint(150, 200))

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    response = generate_resp(messages)

    input_tokens += len(tokenizer.apply_chat_template(messages))
    output_tokens += len(tokenizer.encode(response))

total_time = (datetime.now() - start_time).total_seconds()

print(f"Total Time: {total_time} seconds")
print(f"Input Tokens: {input_tokens}")
print(f"Output Tokens: {output_tokens}")
print(f"Total Tokens: {input_tokens + output_tokens}")

  0%|          | 0/30 [00:00<?, ?it/s]

Total Time: 97.873136 seconds
Input Tokens: 60
Output Tokens: 1509
Total Tokens: 1569


# 11. vLLM

In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_model_id = "/gdrive/MyDrive/llm_finetune_arabic_news/models"


!nohup vllm serve "{base_model_id}" --dtype=half --gpu-memory-utilization 0.8 --max_lora_rank 64 --enable-lora --lora-modules news-lora="{adapter_model_id}" &


In [ ]:
!tail -n 30 nohup.out

# 12. Testing - Locust

In [ ]:
%%writefile locust.py

import random
import json
from locust import HttpUser, task, between, constant
from transformers import AutoTokenizer
from faker import Faker

fake = Faker('ar')

class CompletionLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        model_id = "news-lora"
        prompt = fake.text(max_nb_chars=random.randint(150, 200))

        message = {
            "model": model_id,
            "prompt": prompt,
            "max_tokens": 512,
            "temperature": 0.3
        }

        llm_response = self.client.post("/v1/completions", json=message)

        if llm_response.status_code == 200:
            with open("./vllm_tokens.txt", "a") as dest:
                dest.write(json.dumps({
                    "prompt": prompt,
                    "response": llm_response.json()["choices"][0]["text"],
                }, ensure_ascii=False) + "\n")


In [ ]:
!locust --headless -f locust.py --host=http://localhost:8000 -u 20 -r 1 -t "60s" --html=locust_results.html

In [ ]:
vllm_tokens = [
    json.loads(line.strip())
    for line in open("./vllm_tokens.txt") if line.strip() != ""
]

In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

total_input_tokens = sum([ len(tokenizer.encode(rec['prompt'])) for rec in vllm_tokens ])
total_output_tokens = sum([ len(tokenizer.encode(rec['response'])) for rec in vllm_tokens ])

print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")

# 13. vLLM & NGROK

In [ ]:
import numpy as np
import transformers
print("NumPy:", np.__version__)
print("Transformers:", transformers.__version__)

In [ ]:
import subprocess, time

proc = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype=half",
    "--gpu-memory-utilization", "0.85",
    "--max_lora_rank", "64",
    "--enable-lora",
    "--lora-modules", "news-lora=/gdrive/MyDrive/projects/llm_finetune_arabic_news/models"
])

print("Waiting for vLLM to start...")
time.sleep(90)
print("vLLM should be running on http://localhost:8000")

In [ ]:
import subprocess, time

proc = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype=half",
    "--gpu-memory-utilization", "0.85",
    "--max_lora_rank", "64",
    "--enable-lora",
    "--lora-modules", "news-lora=/gdrive/MyDrive/projects/llm_finetune_arabic_news/models"
],
stderr=subprocess.PIPE,
stdout=subprocess.PIPE
)

print("Waiting for vLLM...")
for i in range(50):
    line = proc.stdout.readline()
    if line:
        print(line.decode('utf-8', errors='ignore').strip())

In [ ]:
import requests, time


for i in range(24):
    try:
        r = requests.get("http://localhost:8000/health")
        if r.status_code == 200:
            print("vLLM ready")
            break
    except:
        print(f"still working {(i+1)*5}s")
        time.sleep(5)

In [ ]:
import requests
r = requests.get("http://localhost:8000/health")
print(r.status_code)


In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3B5iZU9vm5V6ZPmhba5LDhRg6ed_7fxZLjadWxECmEhacN2LK")

url = ngrok.connect(8000)
print("vLLM URL:", url)